In [0]:
# ================================================================
# NOTEBOOK: nb_silver_customers
# PURPOSE:  Watermark Incremental + SCD Type 2 Bronze -> Silver
# RUN:      Daily, after ADF pl_ingest_customers_incremental
# SOURCE:   bronze/customers/  (Parquet files appended by ADF)
# TARGET:   silver/customers/  (Delta table with SCD Type 2 history)
# ================================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, when, trim, upper, lower, initcap, to_date, to_timestamp,
    current_timestamp, current_date, datediff, lit, sha2, concat_ws,
    coalesce
)
from pyspark.sql.window import Window


# ================================================================
# 1. PATHS AND SETTINGS
# ================================================================

BRONZE_PATH = (
    "abfss://source@stshopsensedevhj.dfs.core.windows.net/"
    "bronze/customers/"
)

SILVER_PATH = (
    "abfss://source@stshopsensedevhj.dfs.core.windows.net/"
    "silver/customers/"
)

# Only changes in these columns create a new SCD Type 2 version.
SCD_COLUMNS = ["City", "State", "PinCode", "Segment", "IsPrime"]


# ================================================================
# 2. READ BRONZE
# ================================================================

# This reads every Parquet file currently present under bronze/customers/.
# The deduplication step below keeps only the newest record per CustomerID.
bronze_df = spark.read.parquet(BRONZE_PATH)

print("[SCHEMA] Bronze customer schema:")
bronze_df.printSchema()


# ================================================================
# 3. BASIC VALIDATION
# ================================================================

bronze_df = (
    bronze_df
    .filter(col("CustomerID").isNotNull())
    .filter(col("FirstName").isNotNull())
    .filter(col("LastModifiedDate").isNotNull())
)


# ================================================================
# 4. KEEP THE LATEST BRONZE RECORD PER CUSTOMER
# ================================================================

# Keep LastModifiedDate as a timestamp so that two changes on the same day
# can still be ordered correctly.
bronze_df = bronze_df.withColumn(
    "LastModifiedDate",
    to_timestamp(col("LastModifiedDate"))
)

dedup_window = (
    Window
    .partitionBy("CustomerID")
    .orderBy(col("LastModifiedDate").desc())
)

bronze_df = (
    bronze_df
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

print(
    f"[DEDUP] After keeping latest per customer: "
    f"{bronze_df.count()} rows"
)


# ================================================================
# 5. TYPE CASTING AND STANDARDIZATION
# ================================================================

bronze_df = (
    bronze_df
    .withColumn("RegistrationDate", to_date(col("RegistrationDate")))
    .withColumn("CustomerID", upper(trim(col("CustomerID"))))
    .withColumn("FirstName", initcap(trim(col("FirstName"))))
    .withColumn("LastName", initcap(trim(col("LastName"))))
    .withColumn("Email", lower(trim(col("Email"))))
    .withColumn("Phone", trim(col("Phone")))
    .withColumn("City", initcap(trim(col("City"))))
    .withColumn("State", initcap(trim(col("State"))))
    .withColumn("PinCode", trim(col("PinCode")))
    .withColumn("Segment", upper(trim(col("Segment"))))
    .withColumn("IsPrime", upper(trim(col("IsPrime"))))
)


# ================================================================
# 6. BUSINESS-DERIVED COLUMNS
# ================================================================

bronze_df = (
    bronze_df

    # Add a space between first and last name.
    .withColumn(
        "FullName",
        F.concat_ws(" ", col("FirstName"), col("LastName"))
    )

    .withColumn(
        "IsPrimeBool",
        col("IsPrime") == lit("TRUE")
    )

    .withColumn(
        "TenureDays",
        datediff(current_date(), col("RegistrationDate"))
    )

    .withColumn(
        "TenureCategory",
        when(col("TenureDays") <= 90, "NEW")
        .when(col("TenureDays") <= 365, "GROWING")
        .when(col("TenureDays") <= 1095, "ESTABLISHED")
        .otherwise("LOYAL")
    )

    .withColumn(
        "IsHighValue",
        (col("Segment") == lit("VIP")) | (col("IsPrimeBool") == lit(True))
    )

    # Coalesce nulls before hashing so comparisons remain deterministic.
    .withColumn(
        "_scd_hash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(col(c).cast("string"), lit("<NULL>"))
                    for c in SCD_COLUMNS
                ]
            ),
            256
        )
    )
)


# ================================================================
# 7. SCD TYPE 2 PROCESSING
# ================================================================

if not DeltaTable.isDeltaTable(spark, SILVER_PATH):

    # ------------------------------------------------------------
    # INITIAL LOAD
    # ------------------------------------------------------------
    print("[INIT] Silver customers does not exist - creating it")

    initial_silver = (
        bronze_df
        .withColumn("valid_from", col("RegistrationDate"))
        .withColumn("valid_to", lit("9999-12-31").cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn("version_num", lit(1).cast("integer"))
        .withColumn("_silver_load_ts", current_timestamp())
        .withColumn("_source", lit("initial_watermark_load"))
    )

    (
        initial_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(SILVER_PATH)
    )

    print(
        f"[INIT] Silver created: {initial_silver.count()} rows "
        f"(all version 1)"
    )

else:

    # ------------------------------------------------------------
    # SUBSEQUENT LOADS
    # ------------------------------------------------------------
    print("[SCD2] Silver exists - applying SCD Type 2 logic")

    silver_table = DeltaTable.forPath(spark, SILVER_PATH)
    existing_df = silver_table.toDF()

    # Keep only the current Silver version for comparison.
    current_silver = (
        existing_df
        .filter(col("is_current") == lit(True))
        .select(
            col("CustomerID").alias("existing_customer_id"),
            col("_scd_hash").alias("existing_scd_hash"),
            col("version_num").alias("existing_version_num")
        )
    )

    # Explicit join condition avoids ambiguous/hidden alias behavior.
    joined = (
        bronze_df.alias("new")
        .join(
            current_silver.alias("existing"),
            col("new.CustomerID") == col("existing.existing_customer_id"),
            "left"
        )
    )

    # ------------------------------------------------------------
    # CASE 1: COMPLETELY NEW CUSTOMERS
    # ------------------------------------------------------------
    new_customers = (
        joined
        .filter(col("existing.existing_customer_id").isNull())
        .select("new.*")
        .withColumn("valid_from", current_date())
        .withColumn("valid_to", lit("9999-12-31").cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn("version_num", lit(1).cast("integer"))
        .withColumn("_silver_load_ts", current_timestamp())
        .withColumn("_source", lit("watermark_new"))
    )

    # Cache and materialize before changing the Silver Delta table.
    # Spark DataFrames are lazy. Without this, the DataFrame can be
    # recalculated after the old Silver rows are expired and become empty.
    new_customers = new_customers.cache()
    new_count = new_customers.count()
    print(f"[SCD2] New customers found: {new_count}")

    # ------------------------------------------------------------
    # CASE 2: EXISTING CUSTOMERS WITH AN ACTUAL SCD CHANGE
    # ------------------------------------------------------------
    changed_customers = (
        joined
        .filter(col("existing.existing_customer_id").isNotNull())
        .filter(col("new._scd_hash") != col("existing.existing_scd_hash"))
        .select(
            "new.*",
            col("existing.existing_version_num").alias("old_version_num")
        )
        .withColumn(
            "version_num",
            (col("old_version_num") + lit(1)).cast("integer")
        )
        .drop("old_version_num")
        .withColumn("valid_from", current_date())
        .withColumn("valid_to", lit("9999-12-31").cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn("_silver_load_ts", current_timestamp())
        .withColumn("_source", lit("watermark_scd2_change"))
        .cache()
    )

    # This action materializes the changed rows before silver_table.update().
    changed_count = changed_customers.count()

    print("[CHECK] Versions prepared for changed customers:")
    changed_customers.select(
        "CustomerID",
        "version_num"
    ).show(truncate=False)

    changed_ids = [
        row.CustomerID
        for row in changed_customers.select("CustomerID").collect()
    ]

    print(f"[SCD2] Changed customers found: {changed_count}")

    # ------------------------------------------------------------
    # CASE 3: EXISTING CUSTOMERS WITH NO BUSINESS CHANGE
    # ------------------------------------------------------------
    unchanged_count = (
        joined
        .filter(col("existing.existing_customer_id").isNotNull())
        .filter(col("new._scd_hash") == col("existing.existing_scd_hash"))
        .count()
    )

    print(
        f"[SCD2] Unchanged customers skipped: "
        f"{unchanged_count}"
    )

    # ------------------------------------------------------------
    # STEP A: EXPIRE CURRENT SILVER VERSIONS
    # ------------------------------------------------------------
    if changed_ids:
        silver_table.update(
            condition=(
                col("CustomerID").isin(changed_ids)
                & (col("is_current") == lit(True))
            ),
            set={
                "valid_to": F.date_sub(current_date(), 1),
                "is_current": lit(False)
            }
        )

        print(
            f"[SCD2] Expired old versions for "
            f"{len(changed_ids)} customers"
        )

    # ------------------------------------------------------------
    # STEP B: APPEND NEW AND CHANGED VERSIONS
    # ------------------------------------------------------------
    # Write these separately. This makes version_num behavior explicit
    # and avoids an accidental reset while combining two DataFrames.

    if new_count > 0:
        (
            new_customers.write
            .format("delta")
            .mode("append")
            .save(SILVER_PATH)
        )
        print(f"[SCD2] Inserted {new_count} new customers")

    if changed_count > 0:
        print("[CHECK] Changed rows immediately before append:")
        changed_customers.select(
            "CustomerID",
            "version_num"
        ).show(truncate=False)

        (
            changed_customers.write
            .format("delta")
            .mode("append")
            .save(SILVER_PATH)
        )
        print(
            f"[SCD2] Inserted {changed_count} changed versions"
        )

    # Release cached data after all Delta writes finish.
    new_customers.unpersist()
    changed_customers.unpersist()


# ================================================================
# 8. FINAL VERIFICATION
# ================================================================

final_df = spark.read.format("delta").load(SILVER_PATH)

total_count = final_df.count()
current_count = final_df.filter(col("is_current") == lit(True)).count()
historical_count = final_df.filter(col("is_current") == lit(False)).count()

customers_with_history = (
    final_df
    .groupBy("CustomerID")
    .count()
    .filter(col("count") > 1)
    .count()
)

# Important SCD2 quality test:
# Every customer should have exactly one current record.
duplicate_current_customers = (
    final_df
    .filter(col("is_current") == lit(True))
    .groupBy("CustomerID")
    .count()
    .filter(col("count") != 1)
    .count()
)

print("\n[VERIFY] Silver customers final state:")
print(f"    Total rows (all versions): {total_count}")
print(f"    Current versions:          {current_count}")
print(f"    Historical versions:       {historical_count}")
print(f"    Customers with history:    {customers_with_history}")
print(f"    Invalid current customers: {duplicate_current_customers}")

print("\n[SEGMENT BREAKDOWN] Current customers only:")
(
    final_df
    .filter(col("is_current") == lit(True))
    .groupBy("Segment")
    .agg(
        F.count("CustomerID").alias("customer_count"),
        F.sum(
            when(col("IsPrimeBool") == lit(True), 1).otherwise(0)
        ).alias("prime_members")
    )
    .orderBy("customer_count", ascending=False)
    .show()
)

print("[TENURE BREAKDOWN] Current customers only:")
(
    final_df
    .filter(col("is_current") == lit(True))
    .groupBy("TenureCategory")
    .count()
    .orderBy("count", ascending=False)
    .show()
)

display(
    final_df
    .filter(col("is_current") == lit(True))
    .select(
        "CustomerID",
        "FullName",
        "City",
        "State",
        "Segment",
        "IsPrimeBool",
        "TenureDays",
        "TenureCategory",
        "IsHighValue",
        "valid_from",
        "valid_to",
        "is_current",
        "version_num"
    )
    .orderBy("CustomerID")
    .limit(10)
)

[SCHEMA] Bronze customer schema:
root
 |-- CustomerID: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- PinCode: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- IsPrime: string (nullable = true)
 |-- RegistrationDate: date (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)

[DEDUP] After keeping latest per customer: 500 rows
[SCD2] Silver exists - applying SCD Type 2 logic
[SCD2] New customers found: 0
[CHECK] Versions prepared for changed customers:
+----------+-----------+
|CustomerID|version_num|
+----------+-----------+
|CUST00005 |2          |
|CUST00004 |2          |
|CUST00001 |2          |
|CUST00015 |2          |
|CUST00008 |2          |
|CUST00007 |2          |
|CUST00010 |2          |
|CUST00002 |2          |
|CUST00003 |2         

CustomerID,FullName,City,State,Segment,IsPrimeBool,TenureDays,TenureCategory,IsHighValue,valid_from,valid_to,is_current,version_num
CUST00006,Priya Patel,Chennai,Tamil Nadu,REGULAR,true,1697,LOYAL,true,2021-11-20,9999-12-31,true,1
CUST00009,Arjun Singh,Delhi,Delhi,REGULAR,true,2293,LOYAL,true,2020-04-03,9999-12-31,true,1
CUST00011,Sneha Agarwal,Hyderabad,Telangana,REGULAR,false,1659,LOYAL,false,2021-12-28,9999-12-31,true,1
CUST00012,Pooja Singh,Bangalore,Karnataka,PREMIUM,true,1293,LOYAL,true,2022-12-29,9999-12-31,true,1
CUST00013,Arjun Verma,Jaipur,Rajasthan,REGULAR,true,1937,LOYAL,true,2021-03-25,9999-12-31,true,1
CUST00014,Rajesh Kumar,Mumbai,Maharashtra,VIP,false,1740,LOYAL,true,2021-10-08,9999-12-31,true,1
CUST00016,Sneha Reddy,Pune,Maharashtra,REGULAR,false,2101,LOYAL,false,2020-10-12,9999-12-31,true,1
CUST00017,Kavya Nair,Surat,Gujarat,REGULAR,false,1509,LOYAL,false,2022-05-27,9999-12-31,true,1
CUST00018,Rahul Patel,Ahmedabad,Gujarat,REGULAR,false,2200,LOYAL,false,2020-07-05,9999-12-31,true,1
CUST00019,Priya Nair,Kolkata,West Bengal,PREMIUM,false,1165,LOYAL,false,2023-05-06,9999-12-31,true,1


In [0]:
from pyspark.sql.functions import col

changed_ids = [
    "CUST00001", "CUST00002", "CUST00003",
    "CUST00004", "CUST00005", "CUST00007",
    "CUST00008", "CUST00010", "CUST00015"
]

history_df = (
    spark.read.format("delta").load(SILVER_PATH)
    .filter(col("CustomerID").isin(changed_ids))
    .select(
        "CustomerID",
        "City",
        "State",
        "Segment",
        "IsPrimeBool",
        "valid_from",
        "valid_to",
        "is_current",
        "version_num"
    )
    .orderBy("CustomerID", "version_num")
)

display(history_df)

CustomerID,City,State,Segment,IsPrimeBool,valid_from,valid_to,is_current,version_num
CUST00001,Hyderabad,Telangana,PREMIUM,false,2020-02-21,2026-07-13,false,1
CUST00002,Bangalore,Karnataka,VIP,true,2023-10-17,2026-07-13,false,1
CUST00003,Mumbai,Maharashtra,PREMIUM,true,2020-07-10,2026-07-13,false,1
CUST00004,Pune,Maharashtra,PREMIUM,true,2023-08-23,2026-07-13,false,1
CUST00005,Pune,Maharashtra,PREMIUM,true,2020-01-14,2026-07-13,false,1
CUST00007,Ahmedabad,Gujarat,PREMIUM,true,2021-06-25,2026-07-13,false,1
CUST00008,Delhi,Delhi,VIP,true,2023-02-04,2026-07-13,false,1
CUST00010,Bangalore,Karnataka,PREMIUM,true,2022-02-17,2026-07-13,false,1
CUST00015,Jaipur,Rajasthan,PREMIUM,false,2021-03-11,2026-07-13,false,1


In [0]:
from pyspark.sql.functions import col, lit, current_date, current_timestamp

changed_ids = [
    "CUST00001", "CUST00002", "CUST00003",
    "CUST00004", "CUST00005", "CUST00007",
    "CUST00008", "CUST00010", "CUST00015"
]

# Read latest Bronze values for the 9 changed customers
latest_changed = (
    bronze_df
    .filter(col("CustomerID").isin(changed_ids))
    .withColumn("valid_from", current_date())
    .withColumn("valid_to", lit("9999-12-31").cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn("version_num", lit(2).cast("integer"))
    .withColumn("_silver_load_ts", current_timestamp())
    .withColumn("_source", lit("manual_scd2_repair"))
)

# Keep exactly the same schema and column order as Silver
silver_columns = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
    .columns
)

rows_to_append = latest_changed.select(*silver_columns)

print("Rows to append:", rows_to_append.count())

rows_to_append.select(
    "CustomerID",
    "City",
    "State",
    "Segment",
    "IsPrimeBool",
    "is_current",
    "version_num"
).show(truncate=False)

(
    rows_to_append.write
    .format("delta")
    .mode("append")
    .save(SILVER_PATH)
)

print("[FIX] 9 version-2 rows appended")

Rows to append: 9
+----------+---------+-------------+-------+-----------+----------+-----------+
|CustomerID|City     |State        |Segment|IsPrimeBool|is_current|version_num|
+----------+---------+-------------+-------+-----------+----------+-----------+
|CUST00001 |Chennai  |Tamil Nadu   |PREMIUM|false      |true      |2          |
|CUST00002 |Bangalore|Karnataka    |REGULAR|true       |true      |2          |
|CUST00003 |Mumbai   |Maharashtra  |PREMIUM|false      |true      |2          |
|CUST00004 |Kolkata  |West Bengal  |PREMIUM|true       |true      |2          |
|CUST00005 |Pune     |Maharashtra  |VIP    |true       |true      |2          |
|CUST00007 |Delhi    |Delhi        |PREMIUM|true       |true      |2          |
|CUST00008 |Delhi    |Delhi        |VIP    |false      |true      |2          |
|CUST00010 |Bangalore|Karnataka    |VIP    |true       |true      |2          |
|CUST00015 |Lucknow  |Uttar Pradesh|PREMIUM|false      |true      |2          |
+----------+---------+

In [0]:
silver_check = spark.read.format("delta").load(SILVER_PATH)

print("Total rows:", silver_check.count())
print(
    "Current rows:",
    silver_check.filter(col("is_current") == True).count()
)
print(
    "Historical rows:",
    silver_check.filter(col("is_current") == False).count()
)

Total rows: 509
Current rows: 500
Historical rows: 9


In [0]:
display(
    silver_check
    .filter(col("CustomerID").isin(changed_ids))
    .select(
        "CustomerID",
        "City",
        "State",
        "Segment",
        "IsPrimeBool",
        "valid_from",
        "valid_to",
        "is_current",
        "version_num"
    )
    .orderBy("CustomerID", "version_num")
)

CustomerID,City,State,Segment,IsPrimeBool,valid_from,valid_to,is_current,version_num
CUST00001,Hyderabad,Telangana,PREMIUM,false,2020-02-21,2026-07-13,false,1
CUST00001,Chennai,Tamil Nadu,PREMIUM,false,2026-07-14,9999-12-31,true,2
CUST00002,Bangalore,Karnataka,VIP,true,2023-10-17,2026-07-13,false,1
CUST00002,Bangalore,Karnataka,REGULAR,true,2026-07-14,9999-12-31,true,2
CUST00003,Mumbai,Maharashtra,PREMIUM,true,2020-07-10,2026-07-13,false,1
CUST00003,Mumbai,Maharashtra,PREMIUM,false,2026-07-14,9999-12-31,true,2
CUST00004,Pune,Maharashtra,PREMIUM,true,2023-08-23,2026-07-13,false,1
CUST00004,Kolkata,West Bengal,PREMIUM,true,2026-07-14,9999-12-31,true,2
CUST00005,Pune,Maharashtra,PREMIUM,true,2020-01-14,2026-07-13,false,1
CUST00005,Pune,Maharashtra,VIP,true,2026-07-14,9999-12-31,true,2
